# Week 11, Day 2: Fourier series: building signals from harmonics

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w011/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The Gram matrix of harmonics

Sample one period at N = 100 points. Build the matrix `B` whose columns are cos(kt) and sin(kt) for k = 1 to 4 (cosines first), and its Gram matrix `G` = BᵀB of all pairwise dot products. It should be (N/2) times the identity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
N = 100
t = 2 * np.pi * np.arange(N) / N          # one period, endpoint excluded
B = ...
G = ...
print(np.round(G, 6))

In [ ]:
import numpy as np
def test_basis():
    "B has 8 columns: cos and sin of harmonics 1 to 4, sampled at N points"
    assert B.shape == (N, 8)

def test_gram():
    "the Gram matrix B^T B is (N/2) times the identity"
    assert np.allclose(G, (N / 2) * np.eye(8), atol=1e-9)

test_basis()
test_gram()
print("All checks passed")

### Exercise: Fourier coefficients of a square wave and a spike train

Write `fourier_coeffs(x, K)` returning the cosine and sine coefficients for harmonics 1 to K. Apply it to a square wave (+1 for the first half period, −1 for the second) sampled at 2000 points, with K = 10, and compare `b_sq` with $4/(\pi k)$ for odd k. Then look at the coefficients of a regular spike train.

In [ ]:
def fourier_coeffs(x, K):
    n = len(x)
    tt = 2 * np.pi * np.arange(n) / n
    a = np.array([...])
    b = np.array([...])
    return a, b

M = 2000
t = 2 * np.pi * np.arange(M) / M
square = np.where(t < np.pi, 1.0, -1.0)
a_sq, b_sq = fourier_coeffs(square, 10)

spikes = np.zeros(M); spikes[::200] = 1.0        # ten evenly spaced spikes per period
a_sp, b_sp = fourier_coeffs(spikes, 40)

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].stem(range(1, 11), b_sq); ax[0].plot(range(1, 11), [4 / (np.pi * k) if k % 2 else 0 for k in range(1, 11)], "rx"); ax[0].set_title("square wave: b_k"); ax[0].set_xlabel("harmonic k")
ax[1].stem(range(1, 41), np.hypot(a_sp, b_sp)); ax[1].set_title("spike train: amplitude"); ax[1].set_xlabel("harmonic k")
plt.tight_layout(); plt.show()
print("square wave b_k:", b_sq.round(3))

In [ ]:
import numpy as np
def test_fn():
    "fourier_coeffs returns arrays a and b of length K, by projection"
    a, b = fourier_coeffs(np.cos(3 * t), 5)
    assert np.isclose(a[2], 1) and np.allclose(np.delete(a, 2), 0, atol=1e-9) and np.allclose(b, 0, atol=1e-9)

def test_square():
    "the square wave has b_k = 4 / (pi k) for odd k and 0 for even k"
    k = np.arange(1, 11)
    expected = np.where(k % 2 == 1, 4 / (np.pi * k), 0)
    assert np.allclose(b_sq, expected, atol=0.02) and np.allclose(a_sq, 0, atol=0.02)

test_fn()
test_square()
print("All checks passed")

### Exercise: Partial sums, Gibbs and Parseval

Write `reconstruct(a, b, n)` that sums the harmonics at n sample points (no constant term needed for the square wave). Find the maximum of the square wave's partial sums with 25 and 101 harmonics (`overshoots`). Then check Parseval: compare the mean square of the square wave (`power_signal`) with half the sum of squared coefficients up to harmonic 500 (`power_harmonics`).

In [ ]:
def reconstruct(a, b, n):
    tt = 2 * np.pi * np.arange(n) / n
    x = np.zeros(n)
    for k in range(1, len(a) + 1):
        x += ...
    return x

overshoots = []
for K in [25, 101]:
    a, b = fourier_coeffs(square, K)
    overshoots.append(reconstruct(a, b, M).max())

a5, b5 = fourier_coeffs(square, 500)
power_signal = np.mean(square ** 2)
power_harmonics = ...

a, b = fourier_coeffs(square, 25)
plt.figure(figsize=(6, 3)); plt.plot(t, square, "k--", lw=1); plt.plot(t, reconstruct(a, b, M), lw=1.5); plt.xlim(2.5, 3.8)
plt.title("25 harmonics near the jump"); plt.show()
print("overshoots:", np.round(overshoots, 4), "  power: signal", power_signal, " harmonics", round(power_harmonics, 4))

In [ ]:
import numpy as np
def test_recon():
    "reconstruct rebuilds the signal from a and b"
    a, b = fourier_coeffs(np.cos(2 * t) + 0.5 * np.sin(5 * t), 6)
    assert np.allclose(reconstruct(a, b, M), np.cos(2 * t) + 0.5 * np.sin(5 * t), atol=1e-9)

def test_gibbs():
    "the partial sum peaks near 1.18 (9% of the jump of 2 above 1) for 25 and 101 harmonics"
    assert all(abs(o - 1.18) < 0.02 for o in overshoots)

def test_parseval():
    "power in 500 harmonics accounts for more than 99.5% of the square wave's power"
    assert power_harmonics / power_signal > 0.995

test_recon()
test_gibbs()
test_parseval()
print("All checks passed")